In [ ]:
"""
Step 3b - Validate & Clean (Bronze -> Silver)
Customer Health & Sentiment Scoring Multi-Agent AI Solution

Every Bronze row ends up in exactly ONE place:
  data/silver/<table>.parquet       clean rows (with _dq_fix / _dq_warn audit lists)
  data/quarantine/<table>.parquet   rejected rows (with _dq_reject reasons)

Rule types
  REJECT -> row is quarantined (cannot be trusted or joined)
            null/blank key, unparseable or out-of-window timestamp, orphan customer_id,
            exact duplicate, duplicate primary key, invalid critical category, null amount
  FIX    -> value is repaired, row is kept, fix is recorded
            trim/lower-case text, '' -> NULL, Y/N -> BOOLEAN, out-of-range score -> NULL,
            month re-derived from timestamp, invalid context category -> 'unknown',
            missing consent -> FALSE (conservative outreach guardrail)
  WARN   -> row kept unchanged, issue recorded for review

Allowed-value checks use config/allowed_values.json (create it from the file that step3a_profile.py writes).
If it is missing, allowed-value checks are skipped with a warning.

Run:  python step3b_validate_clean.py      (or: python run_pipeline.py --only 3b)

Change log (see decisions.md): D05 ranges from business_rules.RANGES (fixes duration 540 s / session 3,700 s bug and
the never-firing session_over_4h WARN), D06 connection passed explicitly (no module-level global CON), duplicate SQL
helpers removed, D09 notebook-safe finish().
"""

## 1. Imports and configuration

In [ ]:
from __future__ import annotations

import json
from datetime import datetime

import business_rules as br
from common import (BOOLEAN_COLUMNS, BRONZE_DIR, CATEGORICAL, EVENT_TABLES, EVENT_TS,
                    FALSE_TOKENS, PK, QUARANTINE_DIR, REPORT_DIR, SILVER_DIR, TABLES,
                    TRUE_TOKENS, WINDOW_END, WINDOW_START, copy_to_csv, copy_to_parquet, finish,
                    get_connection, load_allowed_values, log, norm, present_columns,
                    register_layer, scalar)

OUT = REPORT_DIR / "dq"

# Categories that drive scoring logic: an unknown value cannot be interpreted -> REJECT.
# All other categoricals (mainly profile/context) are mapped to 'unknown' -> FIX.
CRITICAL_CATEGORICALS = {
    "interactions": ["interaction_type"],
    "transactions": ["transaction_type", "transaction_status"],
    "engagements": ["event_type"],
}

TS_IN_WINDOW = ("{ts} >= TIMESTAMP '" + WINDOW_START + "' AND {ts} < TIMESTAMP '"
                + WINDOW_END + "' + INTERVAL 1 DAY")


def lit(s: str) -> str:
    """SQL string literal."""
    return "'" + s.replace("'", "''") + "'"

## 2. Rule builder (REJECT / FIX / WARN expressions per table)

In [ ]:
def build_spec(con, t: str, allowed: dict | None) -> dict:
    """Returns {'select': {col: expr}, 'reject': [...], 'fix': [...], 'warn': [...]}.
    Conditions are evaluated on the RAW Bronze columns (alias r)."""
    sel: dict[str, str] = {}
    reject: list[tuple[str, str]] = []
    fix: list[tuple[str, str]] = []
    warn: list[tuple[str, str]] = []

    cols = [r[0] for r in con.execute(f"DESCRIBE SELECT * FROM v_bronze_{t}").fetchall()]
    for c in cols:                               # default: pass through
        sel[c] = f"r.{c}"

    # --- keys -------------------------------------------------------------
    for key in {PK[t], "customer_id"}:
        sel[key] = f"NULLIF(trim(r.{key}), '')"
        reject.append((f"null_{key}", f"NULLIF(trim(r.{key}), '') IS NULL"))
        warn.append((f"{key}_had_whitespace", f"r.{key} <> trim(r.{key})"))

    # --- categoricals: normalise + allowed values -------------------------
    bool_cols = BOOLEAN_COLUMNS.get(t, [])
    for c in present_columns(con, f"v_bronze_{t}", CATEGORICAL[t]):
        if c in bool_cols:
            continue
        n = norm(f"r.{c}")
        sel[c] = n
        warn.append((f"{c}_whitespace_trimmed", f"r.{c} <> trim(r.{c})"))
        if allowed and c in allowed.get(t, {}):
            bad = f"{n} IS NOT NULL AND NOT list_contains({br.sql_list(allowed[t][c])}, {n})"
            if c in CRITICAL_CATEGORICALS.get(t, []):
                reject.append((f"{c}_not_allowed", bad))
            elif c == "sentiment_label":
                sel[c] = f"CASE WHEN {bad} THEN NULL ELSE {n} END"
                fix.append((f"{c}_not_allowed_set_null", bad))
            else:
                sel[c] = f"CASE WHEN {bad} THEN 'unknown' ELSE {n} END"
                fix.append((f"{c}_not_allowed_set_unknown", bad))
        if c in CRITICAL_CATEGORICALS.get(t, []):
            reject.append((f"null_{c}", f"{n} IS NULL"))

    # --- booleans ---------------------------------------------------------
    for c in present_columns(con, f"v_bronze_{t}", bool_cols):
        n = norm(f"r.{c}")
        is_t = f"COALESCE(list_contains({br.sql_list(TRUE_TOKENS)}, {n}), FALSE)"
        is_f = f"COALESCE(list_contains({br.sql_list(FALSE_TOKENS)}, {n}), FALSE)"
        if c == "marketing_consent":   # consent guardrail: unknown consent = no consent
            sel[c] = f"CASE WHEN {is_t} THEN TRUE ELSE FALSE END"
            fix.append(("marketing_consent_defaulted_false", f"NOT {is_t} AND NOT {is_f}"))
        else:
            sel[c] = f"CASE WHEN {is_t} THEN TRUE WHEN {is_f} THEN FALSE END"
            fix.append((f"{c}_unparseable_set_null", f"{n} IS NOT NULL AND NOT {is_t} AND NOT {is_f}"))

    # --- numeric ranges (business_rules.RANGES): out-of-range -> NULL --------
    for c, (lo, hi) in br.RANGES.items():
        if c in cols:
            bad = f"r.{c} IS NOT NULL AND (r.{c} < {lo} OR r.{c} > {hi})"
            sel[c] = f"CASE WHEN {bad} THEN NULL ELSE r.{c} END"
            fix.append((f"{c}_out_of_range_set_null", bad))

    # --- survey ratings must be whole numbers (data dictionary scale) -> non-integer = NULL --------
    for c in ("csat_score", "nps_score"):
        if c in cols:
            lo, hi = br.RANGES[c]
            frac = f"r.{c} IS NOT NULL AND r.{c} <> round(r.{c})"
            sel[c] = f"CASE WHEN r.{c} < {lo} OR r.{c} > {hi} OR ({frac}) THEN NULL ELSE r.{c} END"
            fix.append((f"{c}_non_integer_set_null", frac))

    # --- event timestamps and month --------------------------------------
    if t in EVENT_TABLES:
        ts = EVENT_TS[t]
        reject.append((f"null_{ts}", f"r.{ts} IS NULL"))
        reject.append((f"{ts}_out_of_window", f"r.{ts} IS NOT NULL AND NOT ({TS_IN_WINDOW.format(ts='r.' + ts)})"))
        sel["month"] = f"strftime(r.{ts}, '%Y-%m')"
        fix.append(("month_rederived_from_ts", f"r.month IS DISTINCT FROM strftime(r.{ts}, '%Y-%m')"))

    # --- table-specific rules --------------------------------------------
    if t == "customers":
        warn.append(("null_signup_date", "r.signup_date IS NULL"))
        warn.append(("signup_after_window", f"r.signup_date > DATE '{WINDOW_END}'"))
        warn.append(("signup_before_1990", "r.signup_date < DATE '1990-01-01'"))

    if t == "interactions":
        sel["interaction_text"] = "NULLIF(trim(regexp_replace(r.interaction_text, '\\s+', ' ', 'g')), '')"
        warn.append(("empty_interaction_text", "NULLIF(trim(r.interaction_text), '') IS NULL"))
        sel["agent_id"] = "NULLIF(trim(r.agent_id), '')"

    if t == "transactions":
        # keep the allowed-value result of the categorical loop (invalid -> 'unknown'), then upper-case
        sel["currency"] = f"upper({sel['currency']})" if "currency" in sel else "upper(NULLIF(trim(r.currency), ''))"
        reject.append(("null_amount", "r.amount IS NULL"))
        warn.append(("zero_amount", "r.amount = 0"))
        # sign is NOT used to classify refunds in Step 3c; a negative non-refund is reviewed, not reclassified
        warn.append(("negative_amount_non_refund",
                     f"r.amount < 0 AND NOT {br.isin(norm('r.transaction_type'), br.TX_REFUND_TYPES)}"))
        sel["discount_amount"] = ("CASE WHEN r.discount_amount IS NULL OR r.discount_amount < 0 THEN 0 "
                                  "WHEN r.discount_amount > abs(r.amount) THEN abs(r.amount) "
                                  "ELSE r.discount_amount END")
        fix.append(("discount_null_or_negative_set_0", "r.discount_amount IS NULL OR r.discount_amount < 0"))
        fix.append(("discount_capped_at_amount", "r.discount_amount > abs(r.amount)"))

    if t == "engagements":
        sel["campaign_id"] = "NULLIF(trim(r.campaign_id), '')"
        sel["app_version"] = "NULLIF(trim(r.app_version), '')"
        warn.append(("session_over_4h", f"r.session_duration_sec > {br.ENG_LONG_SESSION_WARN_SEC} "
                                        f"AND r.session_duration_sec <= {br.RANGES['session_duration_sec'][1]}"))
        warn.append(("app_version_on_non_app_channel",
                     f"NULLIF(trim(r.app_version), '') IS NOT NULL AND {norm('r.engagement_channel')} <> 'mobile_app'"))

    return {"select": sel, "reject": reject, "fix": fix, "warn": warn}


def flags(rules: list[tuple[str, str]]) -> str:
    """List of rule codes whose condition is TRUE (list_distinct drops the NULLs)."""
    if not rules:
        return "CAST([] AS VARCHAR[])"
    items = ", ".join(f"CASE WHEN COALESCE({cond}, FALSE) THEN {lit(code)} END" for code, cond in rules)
    return f"list_distinct([{items}])"

## 3. Pipeline per table (pass 1 row rules -> pass 2 duplicates / FK -> Silver + Quarantine)

In [ ]:
def clean_table(con, t: str, allowed: dict | None) -> dict:
    log(f"Validating {t}")
    spec = build_spec(con, t, allowed)
    biz_cols = [c for c in spec["select"] if not c.startswith("_")]
    ts_col = EVENT_TS.get(t, "signup_date")

    # Pass 1 - row-level rules
    select_sql = ",\n               ".join(f"{e} AS {c}" for c, e in spec["select"].items())
    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE _p1_{t} AS
        SELECT {select_sql},
               {flags(spec['reject'])} AS _dq_reject,
               {flags(spec['fix'])}    AS _dq_fix,
               {flags(spec['warn'])}   AS _dq_warn
        FROM v_bronze_{t} r
    """)

    # Pass 2 - duplicates and foreign key
    fk_join, fk_flag = "", "NULL"
    if t in EVENT_TABLES:
        fk_join = "LEFT JOIN (SELECT customer_id AS _fk FROM silver_customers) fk ON fk._fk = p.customer_id"
        fk_flag = "CASE WHEN p.customer_id IS NOT NULL AND fk._fk IS NULL THEN 'orphan_customer_id' END"
    double_post = "NULL"
    if t == "transactions":   # same customer, time, amount, type but different id -> possible double posting
        double_post = ("CASE WHEN count(*) OVER (PARTITION BY p.customer_id, p.transaction_ts, p.amount, "
                       "p.transaction_type) > 1 THEN 'possible_double_posting' END")

    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE _p2_{t} AS
        WITH d AS (
            SELECT p.*,
                   row_number() OVER (PARTITION BY {", ".join("p." + c for c in biz_cols)}
                                      ORDER BY p._ingested_at)                      AS _rn_exact,
                   {fk_flag}                                                         AS _fk_issue,
                   {double_post}                                                     AS _dp_issue
            FROM _p1_{t} p {fk_join}
        ), k AS (
            SELECT *,
                   row_number() OVER (PARTITION BY {PK[t]}
                                      ORDER BY _rn_exact, len(_dq_reject), _ingested_at DESC,
                                               {ts_col} DESC NULLS LAST)              AS _rn_pk
            FROM d
        )
        SELECT * EXCLUDE (_rn_exact, _rn_pk, _fk_issue, _dp_issue, _dq_reject, _dq_warn),
               list_concat(_dq_reject, list_distinct([
                   CASE WHEN _rn_exact > 1 THEN 'exact_duplicate' END,
                   CASE WHEN _rn_exact = 1 AND _rn_pk > 1 AND {PK[t]} IS NOT NULL THEN 'duplicate_pk' END,
                   _fk_issue]))                                 AS _dq_reject,
               list_concat(_dq_warn, list_distinct([_dp_issue]))  AS _dq_warn
        FROM k
    """)

    silver_out, quar_out = SILVER_DIR / f"{t}.parquet", QUARANTINE_DIR / f"{t}.parquet"
    copy_to_parquet(con, f"SELECT * EXCLUDE (_dq_reject) FROM _p2_{t} WHERE len(_dq_reject) = 0", silver_out)
    copy_to_parquet(con, f"SELECT * FROM _p2_{t} WHERE len(_dq_reject) > 0", quar_out)
    con.execute(f"CREATE OR REPLACE VIEW silver_{t} AS SELECT * FROM read_parquet('{silver_out.as_posix()}')")

    bronze, silver, quarantined = con.execute(f"""
        SELECT (SELECT count(*) FROM v_bronze_{t}),
               count(*) FILTER (WHERE len(_dq_reject) = 0),
               count(*) FILTER (WHERE len(_dq_reject) > 0)
        FROM _p2_{t}""").fetchall()[0]
    log(f"  bronze {bronze:,} -> silver {silver:,} | quarantine {quarantined:,}")
    return {"table": t, "bronze_rows": bronze, "silver_rows": silver,
            "quarantined_rows": quarantined, "reconciles": bronze == silver + quarantined}

## 4. Issue counts and post-checks on Silver (every check must have 0 violations)

In [ ]:
def issue_counts() -> str:
    parts = []
    for t in TABLES:
        for kind in ("reject", "fix", "warn"):
            parts.append(f"""SELECT '{t}' AS table_name, '{kind}' AS rule_type, code AS rule, count(*) AS rows
                             FROM (SELECT unnest(_dq_{kind}) AS code FROM _p2_{t}) GROUP BY ALL""")
    return " UNION ALL ".join(parts) + " ORDER BY table_name, rule_type, rows DESC"


def post_checks(con) -> list[dict]:
    a_lo, a_hi = br.RANGES["age"]
    checks = [
        ("customers PK unique", "SELECT count(*) - count(DISTINCT customer_id) FROM silver_customers"),
        ("customers consent never null", "SELECT count(*) FROM silver_customers WHERE marketing_consent IS NULL"),
        ("customers age range", f"SELECT count(*) FROM silver_customers WHERE age NOT BETWEEN {a_lo} AND {a_hi}"),
    ]
    for t in EVENT_TABLES:
        ts = EVENT_TS[t]
        checks += [
            (f"{t} PK unique", f"SELECT count(*) - count(DISTINCT {PK[t]}) FROM silver_{t}"),
            (f"{t} FK to customers", f"SELECT count(*) FROM silver_{t} e ANTI JOIN silver_customers c USING (customer_id)"),
            (f"{t} ts in window", f"SELECT count(*) FROM silver_{t} WHERE NOT ({TS_IN_WINDOW.format(ts=ts)})"),
            (f"{t} month = ts month", f"SELECT count(*) FROM silver_{t} WHERE month <> strftime({ts}, '%Y-%m')"),
        ]
    for c, tbl in (("duration_sec", "interactions"), ("session_duration_sec", "engagements"),
                   ("pages_viewed", "engagements"), ("sentiment_score_gt", "interactions")):
        lo, hi = br.RANGES[c]
        checks.append((f"{c} {lo}..{hi}", f"SELECT count(*) FROM silver_{tbl} WHERE {c} NOT BETWEEN {lo} AND {hi}"))
    checks += [
        ("csat 1-5", f"SELECT count(*) FROM silver_interactions WHERE csat_score NOT BETWEEN {br.CSAT_MIN} AND {br.CSAT_MAX}"),
        ("nps 0-10", f"SELECT count(*) FROM silver_interactions WHERE nps_score NOT BETWEEN {br.NPS_MIN} AND {br.NPS_MAX}"),
        ("csat integer", "SELECT count(*) FROM silver_interactions WHERE csat_score <> round(csat_score)"),
        ("nps integer", "SELECT count(*) FROM silver_interactions WHERE nps_score <> round(nps_score)"),
        ("discount 0..|amount|", "SELECT count(*) FROM silver_transactions WHERE discount_amount < 0 OR discount_amount > abs(amount)"),
    ]
    out = []
    for name, sql in checks:
        n = scalar(con, sql)
        out.append({"check": name, "violations": n, "status": "PASS" if n == 0 else "FAIL"})
    return out

## 5. Main

In [ ]:
def main() -> bool:
    for d in (SILVER_DIR, QUARANTINE_DIR, OUT):
        d.mkdir(parents=True, exist_ok=True)
    allowed = load_allowed_values()
    if allowed is None:
        log("WARNING: config/allowed_values.json not found - allowed-value checks SKIPPED. "
            "Run step3a_profile.py, review config/allowed_values_suggested.json and save it as "
            "config/allowed_values.json.")

    con = get_connection()
    try:
        register_layer(con, BRONZE_DIR, "v_bronze")
        results = [clean_table(con, t, allowed) for t in TABLES]   # customers first (FK target)
        q_issues = issue_counts()
        print("\nDATA-QUALITY ISSUES BY RULE")
        con.sql(q_issues).show(max_rows=200, max_width=200)
        copy_to_csv(con, q_issues, OUT / "dq_issue_counts.csv")
        checks = post_checks(con)
    finally:
        con.close()

    print("\n" + "=" * 80)
    print(f"{'table':<14}{'bronze':>12}{'silver':>12}{'quarantine':>12}{'reconciles':>12}")
    for r in results:
        print(f"{r['table']:<14}{r['bronze_rows']:>12,}{r['silver_rows']:>12,}"
              f"{r['quarantined_rows']:>12,}{str(r['reconciles']):>12}")
    print("-" * 80)
    for c in checks:
        print(f"  [{c['status']}] {c['check']:<32} violations = {c['violations']:,}")
    ok = all(r["reconciles"] for r in results) and all(c["status"] == "PASS" for c in checks)
    print("=" * 80 + f"\nSTEP 3b STATUS: {'PASSED' if ok else 'FAILED - see reports/dq/'}")

    audit = {"run_at": datetime.now().isoformat(timespec="seconds"),
             "allowed_values_applied": allowed is not None, "ranges": br.RANGES,
             "status": "PASSED" if ok else "FAILED", "tables": results, "post_checks": checks}
    (OUT / f"dq_audit_{datetime.now():%Y%m%d_%H%M%S}.json").write_text(json.dumps(audit, indent=2))
    return ok

In [ ]:
if __name__ == "__main__":
    finish(main())